In [ ]:

# from Kea.utils import plotting

# import matplotlib as mpl
# import matplotlib.pyplot as plt
# from matplotlib.colors import LogNorm
# plotting.modify_rc()

# import pyspedas


In [ ]:

import pyspedas
import h5py
data_names = pyspedas.wind.mfi(
    trange=['2007/01/01', '2007/02/01'],
    datatype='h4-rtn',
    get_support_data=True,
    downloadonly=False)
from pytplot import get_data
data = get_data('BRTN')
br, bt, bn = data.y[:,0], data.y[:,1], data.y[:,2]
t = data.times
f = h5py.File('/nfs/scratch/bishopm1/data/SW/brtn.h5', 'a')
f.create_dataset('BR', data=br)
f.create_dataset('BT', data=bt)
f.create_dataset('BN', data=bn)
f.create_dataset('T', data=t)
f.close()


In [ ]:

import h5py
import numpy as np

f = h5py.File('/nfs/scratch/bishopm1/data/SW/brtn.h5', 'r')

t = f['T'][:]
Br, Bt, Bn = f['BR'][:], f['BT'][:], f['BN'][:]
br, bt, bn = Br - np.nanmean(Br), Bt - np.nanmean(Bt), Bn - np.nanmean(Bn)

dt = np.diff(t)[0]
N = len(br)
T = N*dt
twopi = 2.*np.pi
dw = twopi/T

from scipy.signal import windows

br = windows.tukey(N, 0.01) * br
bt = windows.tukey(N, 0.01) * bt
bn = windows.tukey(N, 0.01) * bn

f.close()

from Kea.statistics.spectra import per_spectra, spectra_base

kDr, fekDr = per_spectra.modal_spectrum(br, phys_dims=(T,))
kDt, fekDt = per_spectra.modal_spectrum(bt, phys_dims=(T,))
kDn, fekDn = per_spectra.modal_spectrum(bn, phys_dims=(T,))

kD = kDr[0]
fekD = fekDr + fekDt + fekDn
fekD = fekD * (dw/twopi)

#k, fek, _ = spectra_base.spectrum_integrate(kD, fekD, spec_type='omni', lenn=(T,), num_bins=N//4, min_bin=dw)
k = kD[N//2:]
fek = (fekD[N//2:] + fekD[0:N//2+1][::-1]) / dw

np.save('kmag.npy', k)
np.save('fek_ii.npy', fek)


In [ ]:

import numpy as np
import time

fek = np.load('fek_ii.npy')

def SmoothySpec(a,start=None,end=None,width=3):
   stime = time.time()
   b=a.copy()
   if start is None: start = 0
   if end is None: end = len(a)
   for i in range(start,end):
      b[i+1::][:-1] = 0.25*b[i::][:-2]+0.5*b[i+1::][:-1]+0.25*b[i::][2:]
      #b[i+1:-1] = 0.25*b[i:-2] + 0.5*b[i+1:-1] + 0.25*b[i+2:]
      if ((i % 1000) == 0) and i != 0:
         etime = time.time()
         print((etime - stime)/i * (end - i)/60./60., 'hr')
   return b

fekss = np.exp(SmoothySpec(np.log(fek)))
np.save('fek_ii_smooth_0.npy', fekss)
for i in range(25):
    fekss = np.exp(SmoothySpec(np.log(fekss)))
    np.save('fek_ii_smooth_%s.npy' % (i+1), fekss)

print('fin.')


In [ ]:

import h5py
import numpy as np

f = h5py.File('/nfs/scratch/bishopm1/data/SW/brtn.h5', 'r')

t = f['T'][:]
Br, Bt, Bn = f['BR'][:], f['BT'][:], f['BN'][:]
br, bt, bn = Br - np.nanmean(Br), Bt - np.nanmean(Bt), Bn - np.nanmean(Bn)

dt = np.diff(t)[0]
N = len(br)
T = N*dt
twopi = 2.*np.pi
dw = twopi/T

from scipy.signal import windows

br = windows.tukey(N, 0.01) * br
bt = windows.tukey(N, 0.01) * bt
bn = windows.tukey(N, 0.01) * bn

f.close()

from Kea.statistics.statfunc import strfn, statfunc_base
from Kea.statistics import statistics_base

grid_dims = (N//2,)
phys_dims = (T,)

lv_pR = statfunc_base.get_all_lagvecs(grid_dims)
sf_pR = strfn.process_lags(br, br, lv_pR, lenn=phys_dims, shape=grid_dims, orders=[2], periodic=False)[0]
np.save('sfr.npy', sf_pR)
#lvm_pR = statfunc_base.get_lagvec_magnitude_array(grid_dims) * dt
#l_pR, sf2_pR, w_pR = statistics_base.bin_data(lvm_pR, sf_pR, bin_func=np.nanmean, cut_excess=True, nan_small=False,
#                                           min_bin=dt, max_bin=T/2., bin_loc='center', log_space=True, num_bins=N//64)

lv_pT = statfunc_base.get_all_lagvecs(grid_dims)
sf_pT = strfn.process_lags(bt, bt, lv_pT, lenn=phys_dims, shape=grid_dims, orders=[2], periodic=False)[0]
np.save('sft.npy', sf_pT)
#lvm_pT = statfunc_base.get_lagvec_magnitude_array(grid_dims) * dt
#l_pT, sf2_pT, w_pT = statistics_base.bin_data(lvm_pT, sf_pT, bin_func=np.nanmean, cut_excess=True, nan_small=False,
#                                           min_bin=dt, max_bin=T/2., bin_loc='center', log_space=True, num_bins=N//64)

lv_pN = statfunc_base.get_all_lagvecs(grid_dims)
sf_pN = strfn.process_lags(bn, bn, lv_pN, lenn=phys_dims, shape=grid_dims, orders=[2], periodic=False)[0]
np.save('sfn.npy', sf_pN)
#lvm_pN = statfunc_base.get_lagvec_magnitude_array(grid_dims) * dt
#l_pN, sf2_pN, w_pN = statistics_base.bin_data(lvm_pN, sf_pN, bin_func=np.nanmean, cut_excess=True, nan_small=False,
#                                           min_bin=dt, max_bin=T/2., bin_loc='center', log_space=True, num_bins=N//64)



In [ ]:

import numpy as np

from Kea.statistics.spectra import per_spectra, spectra_base

import h5py

D = 1
twopi = 2.*np.pi
L = twopi
N = 10000
dk = twopi/L

phys_dims = [L for _ in range(D)]
grid_dims = [N for _ in range(D)]

Afile = h5py.File('T.Fr.per.h5', 'r')
Afield = Afile['T.Fr'][:]
Afile.close()

kD, fekD = per_spectra.modal_spectrum(Afield, phys_dims=phys_dims)
fekD = fekD * (dk/twopi)**D

np.save('../out/per_fekD.npy', fekD)
np.save('../out/per_kD.npy', kD)

ko, feko, w = spectra_base.spectrum_integrate(kD, fekD, spec_type='omni', lenn=phys_dims, num_bins=N//2)

np.save('../out/per_ko.npy', ko)
np.save('../out/per_feko.npy', feko)



In [ ]:

import numpy as np

from Kea.statistics.spectra import per_spectra, spectra_base

import h5py

D = 2
twopi = 2.*np.pi
L = twopi
N = 8000
dk = twopi/L

phys_dims = [L for _ in range(D)]
grid_dims = [N for _ in range(D)]

Afile = h5py.File('T.Fr.per.h5', 'r')
Afield = Afile['T.Fr'][:,:]
Afile.close()

kD, fekD = per_spectra.modal_spectrum(Afield, phys_dims=phys_dims)
fekD = fekD * (dk/twopi)**D

np.save('../out/per_fekD.npy', fekD)
np.save('../out/per_kD.npy', kD)

ko, feko, w = spectra_base.spectrum_integrate(kD, fekD, spec_type='omni', lenn=phys_dims)

np.save('../out/per_ko.npy', ko)
np.save('../out/per_feko.npy', feko)



In [ ]:


import numpy as np

from Kea.statistics.spectra import per_spectra, spectra_base

import h5py

D = 3
twopi = 2.*np.pi
L = twopi
N = 512
dk = twopi/L

phys_dims = [L for _ in range(D)]
grid_dims = [N for _ in range(D)]

Afile = h5py.File('T.Fr.per.h5', 'r')
Afield = Afile['T.Fr'][:,:,:]
Afile.close()

kD, fekD = per_spectra.modal_spectrum(Afield, phys_dims=phys_dims)
fekD = fekD * (dk/twopi)**D

np.save('../out/per_fekD.npy', fekD)
np.save('../out/per_kD.npy', kD)

ko, feko, w = spectra_base.spectrum_integrate(kD, fekD, spec_type='omni', lenn=phys_dims)

np.save('../out/per_ko.npy', ko)
np.save('../out/per_feko.npy', feko)



In [ ]:

# data = pyspedas.wind.mfi(
#     trange=['2007/01/01', '2007/02/01'],
#     datatype='h4-rtn',
#     get_support_data=True,
#     downloadonly=False)


In [ ]:

# from pytplot import tplot

# print(data2)
# tplot(data2)


In [ ]:

# from pytplot import get_data

# br_data = get_data('BRTN')


In [ ]:

# import matplotlib.pyplot as plt

# import numpy as np

# fig, ax = plt.subplots(1, 1, figsize=(3., 1.75), dpi=512)

# n = np.arange(0, len(br_data.y[:,0]))

# ax.plot(n, br_data.y[:,0], linewidth=0.25, color='red')
# ax.plot(n, br_data.y[:,1], linewidth=0.25, color='green')
# ax.plot(n, br_data.y[:,2], linewidth=0.25, color='blue')

# ax.yaxis.set_ticks_position('both')
# ax.xaxis.set_ticks_position('both')
# ax.tick_params(axis='y', direction='in')
# ax.tick_params(axis='y', direction='in', which='minor')
# ax.tick_params(axis='x', direction='in')
# ax.tick_params(axis='x', direction='in', which='minor')
# ax.grid(linestyle=':', alpha=0.3, linewidth=0.5, color='gray')
